## 0. Preparación (solo en Google Colab)

Esta celda no hace nada en local. En Colab clona el repositorio, entra en él e instala las dependencias con las versiones del proyecto (necesarias para que el modelo guardado sea compatible con la app). Si Colab pide reiniciar el entorno, reinícialo y vuelve a ejecutar desde aquí.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO_DIR = Path("/content/Chefwise")
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", "-q", "https://github.com/lezti02/Chefwise.git", str(REPO_DIR)])
    os.chdir(REPO_DIR)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "sentence-transformers>=6.1,<7"])

---
# 04. Recomendador por embeddings

> 🟧 **Esta parte es el modelo candidato.** Intenta mejorar al baseline (TF-IDF) que construimos y medimos en el notebook 03.

En el notebook 03 armamos el **Modelo 1 (TF-IDF, el baseline)**, que compara recetas por las **palabras exactas** que comparten.
Aquí armamos el **Modelo 2 (embeddings)**, que intenta comparar recetas por su **significado**.

La pregunta que queremos responder (en el notebook 05) es una sola:

> **¿Los embeddings recomiendan mejor que el baseline? ¿Vale la pena lo que cuestan?**

```text
recipe_text ──► TF-IDF (baseline) ──► números por palabra ────┐
                                                              ├──► comparar ──► las 10 más parecidas ──► calificar
recipe_text ──► embeddings ────────► números por significado ─┘
```

Para que la comparación sea justa, **todo lo demás es igual** para los dos modelos:
el mismo dataset (`recetas_modelo.csv`), el mismo texto (`recipe_text`), las mismas preguntas y la misma forma de calificar (el conjunto de preguntas lo crea el notebook 03 y lo reutiliza el notebook 05).

**Dónde se guarda lo que calcula este notebook:**

| Qué | Dónde |
| --- | --- |
| Embeddings de las recetas (este notebook) | `models/recetas_modelo/` |


## 2.1 ¿Qué es un embedding? 

### La idea

Un **embedding** es una lista de 384 números que un modelo de lenguaje le asigna a un texto.
Ese modelo leyó muchísimo texto antes, y aprendió a darle **números parecidos a textos que significan lo mismo**,
aunque usen palabras distintas.

Piensa en un mapa: cada receta es un punto. Las recetas parecidas quedan **cerca** en el mapa.
TF-IDF también hace un mapa, pero solo junta recetas que **comparten palabras exactas**.

### TF-IDF (baseline) contra embeddings

| | TF-IDF (Modelo 1, baseline) | Embeddings (Modelo 2) |
| --- | --- | --- |
| Qué mira | Qué palabras exactas aparecen | De qué trata el texto |
| `jitomate` y `tomate` | Son palabras distintas: no se parecen nada | Pueden quedar cerca |
| Tamaño de cada receta | Miles de columnas, casi todas en cero | 384 números |
| Qué necesita | Solo nuestros datos; se arma en segundos | Un modelo externo de ~470 MB |
| ¿Se puede explicar? | Sí: se ve qué palabras pesan | No: los 384 números no se leen a ojo |

### ¿Cómo se comparan dos recetas?

Igual que en el notebook 03: con la **similitud coseno**. Si dos recetas apuntan hacia el mismo lado del mapa, el número es alto.

Un detalle: en embeddings casi todos los números salen altos (0.7, 0.8...), incluso entre recetas que no tienen nada que ver.
Por eso **lo que importa es el orden** (cuál queda primero), no el número suelto.

### Lo que ninguno de los dos sabe

Ninguno sabe qué le gusta a cada persona. Los dos miden **parecido entre recetas**, nada más.


## 2.2 Librerías y archivos

* `sentence-transformers`: la librería que convierte texto en embeddings (usa `torch` por dentro).
* `src/embeddings.py`: funciones del proyecto para no repetir código.

El modelo de embeddings ya está descargado en `models/embedding_model/`, así que **no hace falta internet**.
Si no estuviera, la primera vez se baja solo de Hugging Face (~470 MB).

Este notebook se puede correr solo: vuelve a leer el CSV y calcula (o carga) los embeddings.


In [1]:
# --- Herramientas básicas de Python ---
import sys
from pathlib import Path

# --- Datos y tablas ---
import numpy as np
import pandas as pd
from IPython.display import display

# Ubicamos la carpeta principal del proyecto.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists():          # si estamos dentro de notebooks/, subimos un nivel
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

# --- Código propio del proyecto (carpeta src/) ---
# emb: funciones para crear y usar embeddings.
from src import embeddings as emb
from src.exceptions import ArtifactMismatchError, ArtifactNotFoundError

# --- Rutas de archivos y carpetas ---
# CSV con las recetas (el mismo que usó el notebook 03).
DATA_PATH = PROJECT_DIR / "data" / "processed" / "recetas_modelo.csv"
# Carpeta donde este notebook guarda los embeddings (la misma del notebook 03).
MODELS_DIR = PROJECT_DIR / "models" / "recetas_modelo"
# Copia local del modelo de lenguaje (así no hace falta internet).
EMBEDDING_MODEL_DIR = PROJECT_DIR / "models" / emb.EMBEDDING_MODEL_DIRNAME   # copia local del modelo
# Nombre del modelo de embeddings que elegimos.
EMBEDDING_MODEL_NAME = emb.EMBEDDING_MODEL_NAME
# Este modelo pide una "etiqueta" al inicio del texto:
# "query: " para lo que escribe la persona y "passage: " para las recetas.
QUERY_PREFIX, PASSAGE_PREFIX = emb.default_prefixes(EMBEDDING_MODEL_NAME)

# Opciones de cómo se muestran las tablas.
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

# Resumen de lo que vamos a usar y con qué versiones de librerías.
print("Dataset:", DATA_PATH.relative_to(PROJECT_DIR))
print("Modelo de embeddings:", EMBEDDING_MODEL_NAME)
print("Versiones:", emb.library_versions())


Dataset: data/processed/recetas_modelo.csv
Modelo de embeddings: intfloat/multilingual-e5-small
Versiones: {'sentence_transformers': '6.1.0', 'torch': '2.14.1', 'transformers': '5.18.0', 'numpy': '2.5.3'}


## 2.2.1 Cargar el dataset del modelo

La celda funciona igual si Jupyter arranca en la carpeta del proyecto o dentro de `notebooks/`.

Antes de seguir revisamos tres cosas, para no entrenar con datos rotos:

1. que estén las columnas que vamos a usar;
2. que ninguna receta tenga el texto vacío;
3. que no haya recetas repetidas (`recipe_id` único).


In [ ]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists():      # si estamos dentro de notebooks/, subimos un nivel
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

DATA_PATH = PROJECT_DIR / "data" / "processed" / "recetas_modelo.csv"
# Leemos el CSV y lo guardamos en `df`: una tabla con una fila por receta.
df = pd.read_csv(DATA_PATH)

COLUMNAS_NECESARIAS = ["recipe_id", "recipe_key", "name", "ingredients_core", "recipe_text",
                       "country", "difficulty", "total_time_min"]
faltan = [c for c in COLUMNAS_NECESARIAS if c not in df.columns]
# `assert` detiene el notebook con un mensaje claro si algo está mal. Revisamos 3 cosas:
#   1) que no falten columnas, 2) que ninguna receta tenga el texto vacío,
#   3) que no haya recetas repetidas.
assert not faltan, f"Faltan columnas: {faltan}"
assert df["recipe_text"].notna().all() and (df["recipe_text"].str.strip() != "").all(), "hay textos vacíos"
assert df["recipe_id"].is_unique, "hay recetas repetidas"

print("Archivo:", DATA_PATH.relative_to(PROJECT_DIR))
print(f"Recetas: {len(df):,} | columnas: {df.shape[1]}")


Archivo: data/processed/recetas_modelo.csv
Recetas: 28,798 | columnas: 43


In [ ]:
texts = df["recipe_text"].astype(str).tolist()
recipe_ids = df["recipe_id"].to_numpy()
id_to_pos = {int(r): i for i, r in enumerate(recipe_ids)}   # de recipe_id a su fila en la tabla

print(f"Recetas: {len(df):,}")
print("\nEjemplo de recipe_text:\n ", texts[5][:300])


Recetas: 28,798

Ejemplo de recipe_text:
  hallacas venezolanas carne cochino vino papelon alcaparra harina de maiz aceite de onoto cebolla ajoporro cebollin pimenton rojo pimenton verde aji dulce vinagre pastilla de caldo condimento aceite semilla de onoto harina de maiz precocida agua pimenton cortado en tirita aceituna rellena cebolla en 


## 2.3 ¿Qué modelo de embeddings usamos y por qué?

**Lo que necesitábamos:**

1. Que entienda **español** (de México, España y Sudamérica).
2. Que sea bueno para **buscar textos parecidos** (no un chatbot).
3. Que lea el texto **completo** de cada receta, sin cortarlo.

**Opciones que se revisaron:**

| Modelo | Números por texto | Cuánto texto lee | Tamaño | Comentario |
| --- | --- | --- | --- | --- |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | 128 tokens | chico | Rápido, pero lee poco texto |
| `paraphrase-multilingual-mpnet-base-v2` | 768 | 128 tokens | mediano | Lee igual de poco y pesa el doble |
| **`intfloat/multilingual-e5-small`** | **384** | **512 tokens** | **chico** | **Multilingüe, hecho para buscar, lee suficiente, ligero** |
| `intfloat/multilingual-e5-base` | 768 | 512 tokens | mediano | Mejor, pero más lento |
| `BAAI/bge-m3` | 1024 | 8192 tokens | grande | Muy bueno, demasiado pesado para empezar |

**Elegimos `multilingual-e5-small`** porque cumple todo con el menor costo. 

**Detalle de este modelo:** aprendió con unas "etiquetas" al inicio del texto.
Las recetas van con `"passage: "` y lo que escribe la persona con `"query: "`. Eso lo hace `emb.default_prefixes()`.


In [4]:
# Si existe la copia local del modelo, la usamos; si no, se descargaría de Hugging Face (internet, ~470 MB).
modelo_origen = EMBEDDING_MODEL_DIR if EMBEDDING_MODEL_DIR.is_dir() else EMBEDDING_MODEL_NAME
print("Cargando el modelo desde:", modelo_origen)
# Cargamos el modelo de lenguaje en memoria.
model = emb.load_embedding_model(modelo_origen)

# Datos básicos del modelo: cuántos números da por texto, cuánto texto lee
# y en qué dispositivo corre (CPU o tarjeta gráfica).
print("Números por texto (dimensión):", emb.embedding_dimension(model))
print("Texto máximo que lee (tokens):", model.max_seq_length)
print("Dónde corre:", model.device)
print("Prefijos -> persona:", repr(QUERY_PREFIX), "| receta:", repr(PASSAGE_PREFIX))


Cargando el modelo desde: /Users/i./Documents/DIPLOMADO/MODULO_5/chefWise/models/embedding_model


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Números por texto (dimensión): 384
Texto máximo que lee (tokens): 512
Dónde corre: mps:0
Prefijos -> persona: 'query: ' | receta: 'passage: '


### 2.3.1 ¿Cabe todo el texto de cada receta?

El modelo lee como máximo 512 tokens. Contamos cuántos tokens tiene cada `recipe_text` para saber
si alguna receta se cortaría (y perdería el final del texto).


In [5]:
# Contamos los tokens (pedacitos de palabra) de cada receta, tal como los cuenta el modelo.
token_counts = np.array([len(ids) for ids in model.tokenizer(texts, add_special_tokens=True)["input_ids"]])

# Estadísticas: media, mediana, p90 y p99 (el 90 % / 99 % de las recetas mide menos que ese valor) y máximo.
summary = pd.Series({
    "media": token_counts.mean(), "mediana": np.median(token_counts),
    "p90": np.percentile(token_counts, 90), "p99": np.percentile(token_counts, 99), "máximo": token_counts.max(),
}).round(1)
print("Tokens por receta:")
print(summary.to_string(), "\n")
# ¿Cuántas recetas pasan de 128, 256 y 512 tokens? Las que pasen del límite del modelo se cortarían.
for limit in (128, 256, 512):
    share = (token_counts > limit).mean()
    print(f"Recetas con más de {limit:>3} tokens: {int((token_counts > limit).sum()):>6} ({share:6.2%})")


Tokens por receta:
media       42.8
mediana     41.0
p90         61.0
p99         84.0
máximo     157.0 

Recetas con más de 128 tokens:      4 ( 0.01%)
Recetas con más de 256 tokens:      0 ( 0.00%)
Recetas con más de 512 tokens:      0 ( 0.00%)


## 2.4 Calcular los embeddings de todas las recetas (una sola vez)

```text
recipe_text ─► "passage: " + texto ─► modelo ─► 384 números ─► una fila de la tabla
```

* Se procesan de 64 en 64 recetas, para no llenar la memoria.
* Si ya existen en `models/recetas_modelo/` **y** corresponden exactamente a este texto, se cargan y no se recalculan.
  Si el dataset cambió, se detecta (con una "huella" del texto) y se vuelven a calcular.
* El orden de las filas es el mismo del CSV, igual que en TF-IDF.


In [ ]:
# Intentamos cargar los embeddings ya calculados (solo sirven si corresponden al texto actual).
try:
    emb_art = emb.load_embedding_artifacts(
        MODELS_DIR, texts=texts, expected_model_name=EMBEDDING_MODEL_NAME, load_model=False
    )
    print("Embeddings CARGADOS (no se recalcularon).")
# Si no existen o no coinciden con el dataset, hay que calcularlos.
except (ArtifactNotFoundError, ArtifactMismatchError) as motivo:
    print("Se calculan ahora. Motivo:", motivo, "\n")
    # Calculamos los embeddings de todas las recetas (de 64 en 64) y los guardamos para la próxima vez.
    emb_art = emb.build_embeddings(texts, model_name=EMBEDDING_MODEL_NAME, model=model, show_progress_bar=False)
    emb.save_embedding_artifacts(emb_art, MODELS_DIR, save_model=False)   # el modelo ya vive en models/embedding_model/
    print("Guardados en", MODELS_DIR.relative_to(PROJECT_DIR))
# Le pegamos el modelo cargado al objeto, para poder convertir textos nuevos más adelante.
emb_art.model = model

# `meta`: datos del cálculo (modelo usado, tiempo, dispositivo...).
# `E`: la tabla de embeddings (una fila por receta, 384 números cada una).
meta = emb_art.metadata
E = emb_art.matrix
print(f"\nTabla de embeddings: {E.shape[0]:,} recetas x {E.shape[1]} números")
print(f"Memoria: {E.nbytes / 1e6:.1f} MB")
print(f"Tiempo para calcularla: {meta['build_seconds']} s en '{meta['device']}'")


Se calculan ahora. Motivo: Los embeddings guardados no corresponden al `recipe_text` del CSV actual (cambió el contenido o el orden). Regenéralos. 

Guardados en models/recetas_modelo

Tabla de embeddings: 28,798 recetas x 384 números
Memoria: 44.2 MB
Tiempo para calcularla: 25.82 s en 'mps:0'


## 2.5 Las funciones para recomendar

| Función | Qué hace |
| --- | --- |
| `recomendar_similares(recipe_id)` | receta → las más parecidas |
| `recomendar_por_texto(texto)` | ingredientes escritos → recetas |

Se compara **una** receta contra todas (nunca todas contra todas)
y la receta de partida no se recomienda a sí misma.


In [7]:
# Columnas que mostramos en cada recomendación.
RESULT_COLUMNS = ["recipe_id", "name", "country", "difficulty", "total_time_min"]


def _top_n(similitudes, n, excluir=None):
    """Ordena de más a menos parecida. La receta en la posición `excluir` no entra."""
    sims = np.asarray(similitudes, dtype=np.float64).copy()
    if excluir is not None:
        sims[excluir] = -np.inf
    mejores = np.argsort(-sims, kind="stable")[:n]
    resultado = df.iloc[mejores][RESULT_COLUMNS].copy()
    resultado["similitud"] = np.asarray(similitudes)[mejores].round(3)
    return resultado.reset_index(drop=True)


def _posicion(recipe_id):
    if recipe_id not in id_to_pos:
        raise ValueError(f"No existe recipe_id={recipe_id}")
    return id_to_pos[recipe_id]


def _revisar_texto(texto):
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe al menos una palabra.")


def recomendar_similares(recipe_id, n=10):
    pos = _posicion(recipe_id)
    return _top_n(emb_art.similarities_to_row(pos), n, excluir=pos)


def recomendar_por_texto(texto, n=10):
    _revisar_texto(texto)
    return _top_n(emb_art.similarities(texto), n)


## 2.6 Primera prueba (a ojo)

### 2.6.1 Una receta de tacos

Pedimos las 10 recetas más parecidas con embeddings.


In [8]:
# La misma receta de tacos que usa el notebook 03.
es_tacos = df["name"].str.contains(r"\btacos?\b", case=False) & df["country"].eq("México")
EJEMPLO_ID = int(df.loc[es_tacos, "recipe_id"].iloc[0])
pos_ej = id_to_pos[EJEMPLO_ID]

print("Receta de partida:", df.loc[pos_ej, "name"], "|", df.loc[pos_ej, "country"])
display(recomendar_similares(EJEMPLO_ID, n=10))


Receta de partida: Tacos de Cerdo con Salsa Roja | México


,recipe_id,name,country,difficulty,total_time_min,similitud
0,6294,Cerdo en Salsa Verde en Tacos de Tortilla de Maíz,México,reto,120.0,0.953
1,5130,Tacos de Milanesa,México,facil,90.0,0.947
2,6362,Tacos de Filete con Tocino,México,facil,20.0,0.946
3,7964,Tacos de Camaron y Pulpo,México,intermedio,45.0,0.946
4,3822,Tacos Enchilados de Carnitas,México,facil,270.0,0.945
5,6454,Tacos Ahogados,México,facil,40.0,0.945
6,231,Tacos al Pastor,México,facil,45.0,0.944
7,37,Tacos de Carnitas,México,intermedio,90.0,0.943
8,14,Tacos de Carne Molida,México,facil,30.0,0.943
9,4537,Receta de Tacos Norteños,México,facil,40.0,0.943


### 2.6.2 Buscar escribiendo ("con lo que tengo")

Probamos una lista de ingredientes y una frase normal, como la escribiría una persona.


In [9]:
for consulta in ["pollo jitomate chile", "algo dulce con chocolate para el postre"]:
    print(f"########  Consulta: {consulta!r}")
    display(recomendar_por_texto(consulta, n=8)[["recipe_id", "name", "country", "similitud"]])


########  Consulta: 'pollo jitomate chile'


,recipe_id,name,country,similitud
0,9177,Chiles Rellenos de Pollo en Salsa de Tomate,México,0.883
1,14229,Chile Relleno de Queso Bañado en Salsa de Jitomate Roja,México,0.882
2,10829,Chile Relleno con Queso,México,0.882
3,6525,Mixiote de Pollo Express,México,0.882
4,15029,Pollo con Chipotle,México,0.880
5,12204,Pollo a los 3 Chiles,México,0.880
6,9204,Pollo con Chile Piquin,México,0.878
7,5268,Tostitos de Tinga,México,0.878


########  Consulta: 'algo dulce con chocolate para el postre'


,recipe_id,name,country,similitud
0,949,Pastel de Chocolate Humedo y Esponjoso,España,0.894
1,13320,Brownie de Chocolate y Fresa sin Gluten,Internacional,0.892
2,13591,Pastel para los Amantes del Chocolate,México,0.891
3,13725,Chocolatines de Frutos Rojos,México,0.888
4,2610,Turron de Chocolate Blanco con Galletas de Chocolate Rel...,España,0.888
5,141,Pastel de Chocolate con Fresas y Crema Batida,Internacional,0.886
6,10500,Helado de Pistache y Chocolate,México,0.885
7,8708,Pastel de Chocolate y Café,México,0.885


### 2.6.3 ¿Entienden sinónimos?

Pares de palabras o frases. Esto **no cuenta** para la calificación; solo muestra cómo se comporta el embedding:

* En embeddings los números siempre salen altos. Para leerlos, compáralos con el parecido de **dos recetas al azar**
  (última línea): lo que importa es cuánto se separa cada par de ese valor.
* La comparación lado a lado con TF-IDF está en el notebook 05.


In [10]:
pairs = [
    ("jitomate", "tomate"), ("frijoles", "alubias"), ("papa", "patata"),
    ("tacos de pollo", "quesadillas de pollo"), ("sopa de lentejas", "guiso de legumbres"),
    ("pastel de chocolate", "brownie"), ("pollo asado", "pastel de chocolate"),
]


def _sim_emb(a, b):
    va, vb = emb.encode_texts(model, [a, b], prefix=PASSAGE_PREFIX)
    return float(va @ vb)


rng = np.random.default_rng(0)
random_pairs = rng.integers(0, len(df), size=(2000, 2))
rand_emb = (E[random_pairs[:, 0]] * E[random_pairs[:, 1]]).sum(axis=1)

demo = pd.DataFrame(
    [(a, b, _sim_emb(a, b)) for a, b in pairs],
    columns=["frase A", "frase B", "coseno embeddings"],
).round(3)
display(demo)
print(f"Similitud media entre 2000 pares de RECETAS al azar -> embeddings: {rand_emb.mean():.3f}")


,frase A,frase B,coseno embeddings
0,jitomate,tomate,0.933
1,frijoles,alubias,0.883
2,papa,patata,0.912
3,tacos de pollo,quesadillas de pollo,0.960
4,sopa de lentejas,guiso de legumbres,0.928
5,pastel de chocolate,brownie,0.857
6,pollo asado,pastel de chocolate,0.890


Similitud media entre 2000 pares de RECETAS al azar -> embeddings: 0.895
